# 🫀 PULSE ECG — Training (No Google Drive Needed)
**Runtime → Change runtime type → GPU (T4)** before running!

This notebook bypasses Google Drive completely to avoid authentication errors. It will:
1. Clone your code directly from GitHub.
2. Download the datasets directly into Colab from PhysioNet.
3. Train the model and package the output for download.

In [ ]:
# 1️⃣ Clone Project from GitHub
!git clone https://github.com/Gowtham-1301/TCN_BiLSTM.git /content/project
%cd /content/project

import os
for d in ['data/processed', 'models/tfjs_model', 'logs', 'evaluation', 'checkpoints']:
    os.makedirs(d, exist_ok=True)
print('\n✅ Project code cloned successfully!')

In [ ]:
# 2️⃣ Install Dependencies
!pip install -q wfdb scipy scikit-learn matplotlib seaborn tqdm PyYAML h5py pandas tensorboard
print('\n✅ Dependencies installed')

In [ ]:
# 3️⃣ Verify GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    !nvidia-smi
    print('\n✅ GPU is ready!')
else:
    print('\n❌ No GPU detected! Go to: Runtime → Change runtime type → GPU (T4)')

---
## 4️⃣ Download Datasets from PhysioNet
This will take about **30 seconds to 1 minute** as it's downloading MIT-BIH by default for fast training. 
*Uncomment the others if you want the full 14GB training run.*

In [ ]:
%%time
import os

datasets = {
    'mitdb': {
        'url': 'https://physionet.org/files/mitdb/1.0.0/',
        'dest': 'data/raw/mitdb',
        'name': 'MIT-BIH Arrhythmia Database',
        'size': '~230 MB'
    },
    # 'ludb': {
    #     'url': 'https://physionet.org/files/ludb/1.0.1/',
    #     'dest': 'data/raw/ludb',
    #     'name': 'Lobachevsky University DB',
    #     'size': '~3.2 GB'
    # },
    # 'cpsc2018': {
    #     'url': 'https://physionet.org/files/cpsc2018/1.0.0/',
    #     'dest': 'data/raw/CPSC-2018',
    #     'name': 'CPSC 2018',
    #     'size': '~2.1 GB'
    # },
    # 'ptbxl': {
    #     'url': 'https://physionet.org/files/ptb-xl/1.0.3/',
    #     'dest': 'data/raw/PTB-XL',
    #     'name': 'PTB-XL',
    #     'size': '~8.5 GB'
    # },
}

for key, ds in datasets.items():
    dest = ds['dest']
    os.makedirs(dest, exist_ok=True)

    existing_files = sum(1 for _ in os.walk(dest) for f in _[2])
    if existing_files > 10:
        print(f'\n✅ {ds["name"]} already downloaded!')
        continue

    print(f'\n📥 Downloading {ds["name"]} ({ds["size"]})...')
    !wget -r -N -c -np -nH --cut-dirs=3 --no-check-certificate -q --show-progress '{ds["url"]}' -P '{dest}'

print('\n🎉 Datasets ready!')

In [ ]:
# 5️⃣ Train the Model
!python train.py --skip-download

In [ ]:
# 6️⃣ View Results
import json, glob, os
from IPython.display import Image, display

for ef in sorted(glob.glob('evaluation/*.json')):
    print(f'\n📊 {os.path.basename(ef)}')
    with open(ef) as f:
        for k, v in json.load(f).items():
            if isinstance(v, float):
                print(f'  {k}: {v:.4f}')

for pf in sorted(glob.glob('evaluation/*.png')):
    display(Image(filename=pf, width=500))

In [ ]:
# 7️⃣ Export to TensorFlow.js
!pip install -q tensorflowjs
!python convert_to_tfjs.py --metadata
print('\n✅ Converted to TF.js format.')

In [ ]:
# 8️⃣ Download the Finished Files
import shutil
from google.colab import files

shutil.make_archive('/content/pulse_trained_output', 'zip', '/content/project', 'models')
print('📦 Downloading models to your computer...')
files.download('/content/pulse_trained_output.zip')